In [ ]:
!pip install -qq rank_bm25

In [ ]:
import pandas as pd
import zipfile
from rich import print
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

In [ ]:
zip_file_path = '/content/arXiv Paper Abstracts.zip'
csv_file_name = 'arxiv_data.csv' # The name of the CSV file inside the ZIP

# Open the ZIP file
with zipfile.ZipFile(zip_file_path, 'r') as z:
    # Read the specific CSV file into a DataFrame
    with z.open(csv_file_name) as f:
        df = pd.read_csv(f)
        df = df.drop_duplicates(subset=['summaries'])
        df = df.reset_index(drop=True)

In [ ]:
df.sample(5)

,titles,summaries,terms
6611,Adapted Center and Scale Prediction: More Stab...,Pedestrian detection benefits from deep learni...,['cs.CV']
23391,Anthropometric clothing measurements from 3D b...,We propose a full processing pipeline to acqui...,"['cs.CV', 'cs.LG']"
30857,Video-Based Inpatient Fall Risk Assessment: A ...,Inpatient falls are a serious safety issue in ...,"['cs.CV', 'cs.LG']"
25687,Quick Learner Automated Vehicle Adapting its R...,It is essential for an automated vehicle in th...,"['cs.LG', 'cs.SY', 'eess.SY']"
10396,Learning to Estimate Hidden Motions with Globa...,Occlusions pose a significant challenge to opt...,['cs.CV']


In [ ]:
print(df.iloc[234]['summaries'])

Convolutional neural networks (CNNs) have been the de facto standard for
nowadays 3D medical image segmentation. The convolutional operations used in
these networks, however, inevitably have limitations in modeling the long-range
dependency due to their inductive bias of locality and weight sharing. Although
Transformer was born to address this issue, it suffers from extreme
computational and spatial complexities in processing high-resolution 3D feature
maps. In this paper, we propose a novel framework that efficiently bridges a
{\bf Co}nvolutional neural network and a {\bf Tr}ansformer {\bf (CoTr)} for
accurate 3D medical image segmentation. Under this framework, the CNN is
constructed to extract feature representations and an efficient deformable
Transformer (DeTrans) is built to model the long-range dependency on the
extracted feature maps. Different from the vanilla Transformer which treats all
image positions equally, our DeTrans pays attention only to a small set of key
positions by introducing the deformable self-attention mechanism. Thus, the
computational and spatial complexities of DeTrans have been greatly reduced,
making it possible to process the multi-scale and high-resolution feature maps,
which are usually of paramount importance for image segmentation. We conduct an
extensive evaluation on the Multi-Atlas Labeling Beyond the Cranial Vault (BCV)
dataset that covers 11 major human organs. The results indicate that our CoTr
leads to a substantial performance improvement over other CNN-based,
transformer-based, and hybrid methods on the 3D multi-organ segmentation task.
Code is available at \def\UrlFont{\rm\small\ttfamily}
\url{https://github.com/YtongXie/CoTr}

In [ ]:
k = 15
# query = """
# Convolutional neural networks (CNNs) are commonly used for 3D medical image segmentation but have limitations in capturing long-range dependencies. Transformers address this issue but are computationally intensive with high-resolution data. We propose CoTr, a framework that combines CNNs with a deformable Transformer (DeTrans) for efficient 3D segmentation. The CNN extracts features, while DeTrans models long-range dependencies with a deformable self-attention mechanism, reducing complexity by focusing on key positions. Evaluations on the Multi-Atlas Labeling Beyond the Cranial Vault (BCV) dataset show that CoTr significantly outperforms existing methods in 3D multi-organ segmentation.
# """
query = """
Convolutional neural networks (CNNs).
"""
print(query) # paraphrased selected summary

Convolutional neural networks (CNNs).

In [ ]:
from openai import AzureOpenAI
from google.colab import userdata
client = AzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))

openai_response = client.chat.completions.create(
    temperature=0.0,
    max_tokens=None,
    messages=[
    {"role": "system", "content": "You are a helpful assistant who generate hypothetical paper abstract"},
    {"role": "user", "content": f"Generate an abstract:Convolutional neural networks (CNNs)."},
    ], model="gpt-4o-2", # cwsgpt35turbo0613, cwsgpt4turbo
    )
query = openai_response.choices[0].message.content

In [ ]:
corpus = df['summaries'].tolist()
tokenized_corpus = list(map(lambda doc: doc.split(" "), corpus))
bm25 = BM25Okapi(tokenized_corpus)
tokenized_query = query.split(" ")
# top_summaries=bm25.get_top_n(tokenized_query, corpus, n=k)
top_indices=bm25.get_top_n(tokenized_query, df.index.tolist(),n=k)
rag1_result = df.iloc[top_indices][['summaries']].to_markdown(index=False)
rag1_result

'| summaries                                                                       |\n|:--------------------------------------------------------------------------------|\n| Deep Convolutional Neural Network (CNN) is a special type of Neural Networks,   |\n| which has shown exemplary performance on several competitions related to        |\n| Computer Vision and Image Processing. Some of the exciting application areas of |\n| CNN include Image Classification and Segmentation, Object Detection, Video      |\n| Processing, Natural Language Processing, and Speech Recognition. The powerful   |\n| learning ability of deep CNN is primarily due to the use of multiple feature    |\n| extraction stages that can automatically learn representations from the data.   |\n| The availability of a large amount of data and improvement in the hardware      |\n| technology has accelerated the research in CNNs, and recently interesting deep  |\n| CNN architectures have been reported. Several inspiring ideas 

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['summaries'])
query_vec = vectorizer.transform([query])
cosine_similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
top_indices = cosine_similarities.argsort()[-k:][::-1]
top_summaries = df.iloc[top_indices]
rag2_result = top_summaries[['summaries']].to_markdown(index=False)
rag2_result

"| summaries                                                                       |\n|:--------------------------------------------------------------------------------|\n| Deep Convolutional Neural Network (CNN) is a special type of Neural Networks,   |\n| which has shown exemplary performance on several competitions related to        |\n| Computer Vision and Image Processing. Some of the exciting application areas of |\n| CNN include Image Classification and Segmentation, Object Detection, Video      |\n| Processing, Natural Language Processing, and Speech Recognition. The powerful   |\n| learning ability of deep CNN is primarily due to the use of multiple feature    |\n| extraction stages that can automatically learn representations from the data.   |\n| The availability of a large amount of data and improvement in the hardware      |\n| technology has accelerated the research in CNNs, and recently interesting deep  |\n| CNN architectures have been reported. Several inspiring ideas 

In [ ]:
openai_response = client.chat.completions.create(
    temperature=0.0,
    max_tokens=None,
    messages=[
    {"role": "system", "content": "You are a helpful assistant who determinates with rag_result from two results is better(meaning retrieve the most similar), be concise and tell me why"},
    {"role": "user", "content": f"This query:'{query}', retrieve two rags results:{{rag1_result:{rag1_result},rag2_result:{rag2_result}}}, which rag is better?"},
    ], model="gpt-4o-2", # cwsgpt35turbo0613, cwsgpt4turbo
    )

In [ ]:
from IPython.display import Markdown
Markdown(openai_response.choices[0].message.content)

The query is about advances in Convolutional Neural Networks (CNNs), focusing on architectures, applications, and challenges. 

**Rag1_result** is better because:
- It provides a comprehensive overview of CNN architectures, including specific examples like AlexNet, VGG, ResNet, EfficientNet, and Vision Transformers, which aligns well with the query's focus on recent advancements in architectures.
- It discusses a wide array of applications beyond traditional image processing, such as medical imaging and autonomous vehicles, which matches the query's mention of diverse applications.
- It addresses challenges like the need for large datasets, computational costs, and interpretability, which are key points in the query.

**Rag2_result** also covers CNNs but lacks the specific focus on recent architectural advancements and the detailed discussion of applications and challenges that are central to the query.